# March 18 Bundle: Anomalous ASAS-SN Fields Audit

This notebook reconstructs ASAS-SN `field` and `camera_field` diagnostics for the March 18 bundle at `output/runs/runs_march18_bundle_all`.

The persisted MALCA result tables are camera-centric and do not carry field-level diagnostics. This notebook therefore reparses the bundled `.dat3` light curves under `bundle_assets/lightcurves/` and joins them back to candidate metadata by source/file stem.

**Conservative interpretation rule:** all rates and anomaly ranks here are within the March 18 vetted/passing candidate bundle only. They are not survey-wide false-positive rates unless a full survey denominator is added separately.

## Parameters

Defaults are intentionally read-only. Set `CACHE_DERIVED_TABLES = True` only if you want derived parquet tables written under the bundle's `analysis/field_anomalies/` directory.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

# Requested notebook parameters.
BUNDLE_ROOT = Path('../..') / 'output/runs/runs_march18_bundle_all'
FULL_SCAN = True
MAX_EXAMPLE_PLOTS = 30
EVENT_WINDOW_SIGMA_MULT = 2.5
MIN_GROUP_POINTS = 30
ROBUST_Z_THRESHOLD = 4.0
CACHE_DERIVED_TABLES = False

# Smoke-test convenience. Ignored when FULL_SCAN=True.
SMOKE_N_FILES = 80
RANDOM_SEED = 20260318

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 220)
warnings.filterwarnings('ignore', category=FutureWarning)

## Helper Functions

These helpers keep the analysis self-contained in this notebook. They do not modify MALCA production code.

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'malca').is_dir():
            return candidate
    return Path.cwd().resolve()


def resolve_bundle_root(raw: Path) -> Path:
    raw = Path(raw).expanduser()
    repo_root = find_repo_root()
    candidates = [
        raw,
        Path.cwd() / raw,
        repo_root / 'output/runs/runs_march18_bundle_all',
        repo_root / raw,
    ]
    seen: set[str] = set()
    for candidate in candidates:
        candidate = candidate.resolve(strict=False)
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate
    checked = '\n'.join(f'  - {c.resolve(strict=False)}' for c in candidates)
    raise FileNotFoundError(f'Could not resolve March 18 bundle root. Checked:\n{checked}')


def source_id_from_path(value: object) -> str:
    if value is None:
        return ''
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', '<na>'}:
        return ''
    return Path(text).stem.strip()


def robust_sigma(values: object) -> float:
    arr = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    med = float(np.nanmedian(arr))
    mad = float(np.nanmedian(np.abs(arr - med)))
    if np.isfinite(mad) and mad > 0:
        return 1.4826 * mad
    if arr.size > 1:
        iqr = float(np.nanpercentile(arr, 75) - np.nanpercentile(arr, 25))
        if np.isfinite(iqr) and iqr > 0:
            return iqr / 1.349
        std = float(np.nanstd(arr, ddof=1))
        if np.isfinite(std) and std > 0:
            return std
    return 0.0


def robust_z(series: pd.Series) -> pd.Series:
    vals = pd.to_numeric(series, errors='coerce').astype(float)
    finite = vals[np.isfinite(vals)]
    if finite.empty:
        return pd.Series(np.nan, index=series.index)
    med = float(np.nanmedian(finite))
    scale = robust_sigma(finite)
    if not np.isfinite(scale) or scale <= 0:
        scale = float(np.nanstd(finite, ddof=1)) if len(finite) > 1 else np.nan
    if not np.isfinite(scale) or scale <= 0:
        return pd.Series(0.0, index=series.index)
    return (vals - med) / scale


def positive_robust_z(series: pd.Series) -> pd.Series:
    return robust_z(series).clip(lower=0).fillna(0.0)


def safe_fraction(num: object, den: object) -> float:
    try:
        num_f = float(num)
        den_f = float(den)
    except Exception:
        return np.nan
    if not np.isfinite(num_f) or not np.isfinite(den_f) or den_f == 0:
        return np.nan
    return num_f / den_f


def top_value_fraction(frame: pd.DataFrame, group_col: str, value_col: str, count_name: str) -> pd.DataFrame:
    counts = frame.groupby([group_col, value_col], observed=True).size().rename(count_name).reset_index()
    if counts.empty:
        return pd.DataFrame(columns=[group_col, f'top_{value_col}', f'{value_col}_dominance_fraction'])
    idx = counts.groupby(group_col, observed=True)[count_name].idxmax()
    top = counts.loc[idx].copy()
    totals = counts.groupby(group_col, observed=True)[count_name].sum().rename('_total')
    top = top.merge(totals, left_on=group_col, right_index=True, how='left')
    top[f'{value_col}_dominance_fraction'] = top[count_name] / top['_total'].replace(0, np.nan)
    top = top.rename(columns={value_col: f'top_{value_col}'})
    return top[[group_col, f'top_{value_col}', f'{value_col}_dominance_fraction']]


def display_section(title: str, text: str | None = None) -> None:
    body = f'## {title}' if text is None else f'## {title}\n\n{text}'
    display(Markdown(body))

## Resolve Bundle and Load Persisted Metadata

The original candidate `path` values point to the production ASAS-SN filesystem and are expected to be stale on this machine. The notebook resolves light curves through `bundle_assets/lightcurves/{source_id}.dat3`.

In [ ]:
BUNDLE_ROOT = resolve_bundle_root(BUNDLE_ROOT)
RESULTS_DIR = BUNDLE_ROOT / 'results'
LIGHTCURVE_DIR = BUNDLE_ROOT / 'bundle_assets/lightcurves'
REVIEW_DB = BUNDLE_ROOT / 'review/review.db'
RUN_PARAMS_PATH = BUNDLE_ROOT / 'run_params.json'
VETTED_PARQUET = RESULTS_DIR / 'lc_events_vetted.parquet'

assert BUNDLE_ROOT.exists(), BUNDLE_ROOT
assert LIGHTCURVE_DIR.exists(), LIGHTCURVE_DIR
assert VETTED_PARQUET.exists(), VETTED_PARQUET

with RUN_PARAMS_PATH.open() as f:
    run_params = json.load(f)

candidates = pd.read_parquet(VETTED_PARQUET).copy()
if 'path' in candidates.columns:
    candidates['source_id'] = candidates['path'].map(source_id_from_path)
elif 'lc_path' in candidates.columns:
    candidates['source_id'] = candidates['lc_path'].map(source_id_from_path)
elif 'asas_sn_id' in candidates.columns:
    candidates['source_id'] = candidates['asas_sn_id'].astype(str)
else:
    raise ValueError('Candidate parquet has no path/lc_path/asas_sn_id column to derive source_id.')

candidate_source_ids = set(candidates['source_id'].dropna().astype(str))
persisted_field_like_cols = [c for c in candidates.columns if 'field' in c.lower() and c != 'banyan_field_prob']
persisted_camera_like_cols = [c for c in candidates.columns if 'camera' in c.lower()]

display(Markdown(f"""
**Bundle root:** `{BUNDLE_ROOT}`  
**Candidate rows:** `{len(candidates):,}`  
**Unique candidate source IDs:** `{len(candidate_source_ids):,}`  
**Run timestamp:** `{run_params.get('timestamp', 'unknown')}`  
**Baseline:** `{run_params.get('baseline_func', 'unknown')}`  
**Trigger mode:** `{run_params.get('trigger_mode', 'unknown')}`  
**Field-level persisted columns:** `{persisted_field_like_cols}`
"""))

candidates.head()

## Optional Review Labels

Review labels are not required for field diagnostics, but they are useful for checking whether suspicious fields concentrate in particular review outcomes or classes.

In [ ]:
def load_review_metadata(review_db: Path) -> pd.DataFrame:
    if not review_db.exists():
        return pd.DataFrame()
    conn = sqlite3.connect(review_db)
    try:
        review_candidates = pd.read_sql_query(
            'SELECT candidate_id, asas_sn_id, lc_path, source_path FROM candidates', conn
        )
        reviews = pd.read_sql_query('SELECT * FROM reviews', conn)
    finally:
        conn.close()
    if review_candidates.empty:
        return pd.DataFrame()
    review_candidates = review_candidates.copy()
    source_from_lc = review_candidates.get('lc_path', pd.Series('', index=review_candidates.index)).map(source_id_from_path)
    source_from_path = review_candidates.get('source_path', pd.Series('', index=review_candidates.index)).map(source_id_from_path)
    source_from_asas = review_candidates.get('asas_sn_id', pd.Series('', index=review_candidates.index)).astype(str).str.strip()
    review_candidates['source_id'] = source_from_lc.where(source_from_lc.ne(''), source_from_path)
    review_candidates['source_id'] = review_candidates['source_id'].where(review_candidates['source_id'].ne(''), source_from_asas)
    if reviews.empty:
        return review_candidates
    return review_candidates.merge(reviews, on='candidate_id', how='left', suffixes=('', '_review'))


review_meta = load_review_metadata(REVIEW_DB)
review_cols = [c for c in ['source_id', 'candidate_id', 'interest_score', 'event_class', 'review_pass', 'status', 'reviewer', 'notes'] if c in review_meta.columns]
review_by_source = review_meta[review_cols].drop_duplicates('source_id') if review_cols else pd.DataFrame()

n_reviewed = int(review_meta['interest_score'].notna().sum()) if 'interest_score' in review_meta else 0
display(Markdown(f'Loaded `{len(review_meta):,}` review candidate rows and `{n_reviewed:,}` reviewed rows.'))
review_by_source.head()

## Parse Bundled `.dat3` and `.raw2` Files Once

`field` is parsed from the token after `/` in raw `cam_field`; `camera_field` is the full `camera_name/field` token.

In [ ]:
ASASSN_COLUMNS = ['JD', 'mag', 'error', 'good_bad', 'camera_num', 'band', 'saturated', 'cam_field']
RAW2_COLUMNS = ['camera_num', 'raw2_median', 'raw2_sig1_low', 'raw2_sig1_high', 'raw2_p90_low', 'raw2_p90_high']


def parse_dat3_file(path: Path) -> tuple[pd.DataFrame, dict]:
    issue = {'source_id': path.stem, 'file': str(path), 'read_ok': True, 'error': '', 'n_rows': 0,
             'malformed_cam_field': 0, 'empty_field': 0, 'nonfinite_photometry': 0}
    try:
        df = pd.read_csv(path, sep=r'\s+', names=ASASSN_COLUMNS, comment='#', engine='python')
    except Exception as exc:
        issue.update(read_ok=False, error=repr(exc))
        return pd.DataFrame(columns=['source_id', *ASASSN_COLUMNS, 'camera_name', 'field', 'camera_field']), issue
    issue['n_rows'] = int(len(df))
    if df.empty:
        return df, issue
    df['source_id'] = path.stem
    for col in ['JD', 'mag', 'error']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in ['good_bad', 'camera_num', 'band', 'saturated']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    cam_field = df['cam_field'].astype('string').fillna('').str.strip()
    split = cam_field.str.split('/', n=1, expand=True)
    df['camera_name'] = split[0].astype('string').fillna('').str.strip()
    df['field'] = split[1].astype('string').fillna('').str.strip() if split.shape[1] > 1 else ''
    df['camera_field'] = np.where(df['field'].astype(str).str.len() > 0,
                                  df['camera_name'].astype(str) + '/' + df['field'].astype(str),
                                  df['camera_name'].astype(str))

    issue['malformed_cam_field'] = int((~cam_field.str.contains('/', regex=False)).sum())
    issue['empty_field'] = int((df['field'].astype(str).str.len() == 0).sum())
    issue['nonfinite_photometry'] = int((~np.isfinite(df[['JD', 'mag', 'error']].to_numpy(dtype=float))).any(axis=1).sum())
    return df, issue


def parse_raw2_file(path: Path) -> tuple[pd.DataFrame, dict]:
    issue = {'source_id': path.stem, 'file': str(path), 'raw2_read_ok': True, 'raw2_error': '', 'raw2_n_rows': 0}
    if not path.exists():
        issue.update(raw2_read_ok=False, raw2_error='missing')
        return pd.DataFrame(columns=['source_id', *RAW2_COLUMNS, 'expected_scatter']), issue
    try:
        raw = pd.read_csv(path, sep=r'\s+', names=RAW2_COLUMNS, comment='#', engine='python')
    except Exception as exc:
        issue.update(raw2_read_ok=False, raw2_error=repr(exc))
        return pd.DataFrame(columns=['source_id', *RAW2_COLUMNS, 'expected_scatter']), issue
    issue['raw2_n_rows'] = int(len(raw))
    raw['source_id'] = path.stem
    for col in RAW2_COLUMNS:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')
    raw['expected_scatter'] = (raw['raw2_sig1_high'] - raw['raw2_sig1_low']) / 2.0
    return raw, issue


dat3_files_all = sorted(LIGHTCURVE_DIR.glob('*.dat3'))
if not FULL_SCAN:
    rng = np.random.default_rng(RANDOM_SEED)
    n = min(SMOKE_N_FILES, len(dat3_files_all))
    dat3_files = sorted(rng.choice(dat3_files_all, size=n, replace=False).tolist()) if n else []
else:
    dat3_files = dat3_files_all

iterator = tqdm(dat3_files, desc='Parsing dat3/raw2') if tqdm is not None else dat3_files
point_frames: list[pd.DataFrame] = []
raw2_frames: list[pd.DataFrame] = []
parse_issues: list[dict] = []
raw2_issues: list[dict] = []

for dat_path in iterator:
    frame, issue = parse_dat3_file(dat_path)
    point_frames.append(frame)
    parse_issues.append(issue)
    raw2_frame, raw2_issue = parse_raw2_file(dat_path.with_suffix('.raw2'))
    raw2_frames.append(raw2_frame)
    raw2_issues.append(raw2_issue)

points = pd.concat(point_frames, ignore_index=True) if point_frames else pd.DataFrame(columns=['source_id', *ASASSN_COLUMNS, 'camera_name', 'field', 'camera_field'])
raw2 = pd.concat(raw2_frames, ignore_index=True) if raw2_frames else pd.DataFrame(columns=['source_id', *RAW2_COLUMNS, 'expected_scatter'])
parse_issues = pd.DataFrame(parse_issues)
raw2_issues = pd.DataFrame(raw2_issues)

for col in ['source_id', 'camera_name', 'field', 'camera_field', 'cam_field']:
    if col in points.columns:
        points[col] = points[col].astype('category')
for col in ['good_bad', 'camera_num', 'band', 'saturated']:
    if col in points.columns:
        points[col] = pd.to_numeric(points[col], errors='coerce').astype('Int16')
for col in ['mag', 'error']:
    if col in points.columns:
        points[col] = pd.to_numeric(points[col], errors='coerce').astype('float32')
if 'JD' in points.columns:
    points['JD'] = pd.to_numeric(points['JD'], errors='coerce').astype('float64')

if not raw2.empty:
    raw2['source_id'] = raw2['source_id'].astype('category')
    raw2['camera_num'] = pd.to_numeric(raw2['camera_num'], errors='coerce').astype('Int16')

display(Markdown(f'Parsed `{len(dat3_files):,}` `.dat3` files into `{len(points):,}` photometry rows. FULL_SCAN=`{FULL_SCAN}`.'))
points.head()

## Reproducibility and Integrity Checks

These checks are intentionally explicit because any field-level conclusion depends on correct local bundle reconstruction.

In [ ]:
discovered_source_ids = set(pd.Series([p.stem for p in dat3_files_all]).astype(str))
parsed_source_ids = set(points['source_id'].astype(str).unique()) if not points.empty else set()
stale_path_exists = int(candidates['path'].map(lambda p: Path(str(p)).exists()).sum()) if 'path' in candidates.columns else np.nan

expected_raw2 = {p.with_suffix('.raw2').name for p in dat3_files_all}
discovered_raw2 = {p.name for p in LIGHTCURVE_DIR.glob('*.raw2')}

integrity_rows = [
    ('candidate_rows', len(candidates), 'Rows in lc_events_vetted.parquet'),
    ('candidate_unique_source_ids', len(candidate_source_ids), 'Unique source IDs derived from candidate paths'),
    ('bundle_dat3_files_total', len(dat3_files_all), 'All bundled .dat3 files'),
    ('bundle_raw2_files_total', len(discovered_raw2), 'All bundled .raw2 files'),
    ('parsed_dat3_files_this_run', len(dat3_files), 'Files parsed in this notebook run'),
    ('parsed_point_rows', len(points), 'Photometry rows parsed in this notebook run'),
    ('candidate_source_ids_missing_dat3', len(candidate_source_ids - discovered_source_ids), 'Candidate IDs with no bundled .dat3'),
    ('dat3_without_candidate_source_id', len(discovered_source_ids - candidate_source_ids), 'Bundled .dat3 files not present in candidates'),
    ('raw2_missing_for_dat3', len(expected_raw2 - discovered_raw2), 'Missing .raw2 files for bundled .dat3'),
    ('original_candidate_paths_exist_locally', stale_path_exists, 'Expected to be 0 for imported bundle paths'),
    ('persisted_non_banyan_field_columns', len(persisted_field_like_cols), str(persisted_field_like_cols)),
    ('persisted_camera_like_columns', len(persisted_camera_like_cols), str(persisted_camera_like_cols[:20])),
    ('duplicate_candidate_source_ids', int(candidates['source_id'].duplicated().sum()), 'Duplicate source_id rows in candidates'),
    ('duplicate_dat3_source_ids', len(dat3_files_all) - len(discovered_source_ids), 'Duplicate .dat3 stems'),
    ('parse_read_failures', int((~parse_issues.get('read_ok', pd.Series(dtype=bool))).sum()) if not parse_issues.empty else 0, 'Unreadable .dat3 files'),
    ('malformed_cam_field_rows', int(parse_issues.get('malformed_cam_field', pd.Series(dtype=int)).sum()) if not parse_issues.empty else 0, 'Rows where cam_field lacks /'),
    ('empty_field_rows', int(parse_issues.get('empty_field', pd.Series(dtype=int)).sum()) if not parse_issues.empty else 0, 'Rows with empty parsed field'),
    ('nonfinite_photometry_rows', int(parse_issues.get('nonfinite_photometry', pd.Series(dtype=int)).sum()) if not parse_issues.empty else 0, 'Rows with non-finite JD/mag/error'),
    ('suspicious_band_values', sorted(set(points['band'].dropna().astype(int)) - {0, 1}) if not points.empty else [], 'Expected ASAS-SN bands are 0=g, 1=V'),
    ('suspicious_camera_num_values', sorted(set(points['camera_num'].dropna().astype(int)) - set(range(1, 50)))[:20] if not points.empty else [], 'Camera nums outside 1..49'),
]
integrity_summary = pd.DataFrame(integrity_rows, columns=['check', 'value', 'note'])
display(integrity_summary)

if not parse_issues.empty:
    display(parse_issues.query('read_ok == False or malformed_cam_field > 0 or empty_field > 0 or nonfinite_photometry > 0').head(30))
if not raw2_issues.empty:
    display(raw2_issues.query('raw2_read_ok == False').head(30))

## Candidate Projection and Point-Level Robust Residuals

Residuals use within-source medians, preferring `source_id + band + camera_num` and falling back to `source_id + band`, then `source_id`. These residuals are diagnostic only; they are not the MALCA GP baseline residuals.

In [ ]:
important_candidate_cols = [
    'source_id', 'path', 'failed_any', 'bad_cameras_filtered', 'excluded_cameras',
    'n_points', 'n_cameras', 'camera_ids', 'camera_min_points', 'camera_max_points',
    'dip_significant', 'jump_significant', 'dip_run_count', 'jump_run_count',
    'dip_max_run_points', 'jump_max_run_points', 'dip_max_run_cameras', 'jump_max_run_cameras',
    'dip_best_t0', 'jump_best_t0', 'dip_best_width_param', 'jump_best_width_param',
    'dip_best_amp', 'jump_best_amp', 'dip_bayes_factor', 'jump_bayes_factor',
    'dip_max_event_prob', 'jump_max_event_prob', 'dip_best_morph', 'jump_best_morph',
]
important_candidate_cols += [c for c in candidates.columns if c.startswith('failed_') and c not in important_candidate_cols]
important_candidate_cols += [c for c in ['vetting_likely_known', 'microlens_match', 'microlens_catalog', 'final_class', 'yso_class'] if c in candidates.columns]
important_candidate_cols = [c for c in important_candidate_cols if c in candidates.columns]
candidate_projection = candidates[important_candidate_cols].drop_duplicates('source_id').copy()

if not review_by_source.empty:
    review_attach = review_by_source[[c for c in review_by_source.columns if c != 'candidate_id']].drop_duplicates('source_id')
    candidate_projection = candidate_projection.merge(review_attach, on='source_id', how='left')

if points.empty:
    raise RuntimeError('No parsed photometry points. Check BUNDLE_ROOT and FULL_SCAN settings.')

if not raw2.empty:
    raw2_small = raw2[['source_id', 'camera_num', 'expected_scatter']].dropna(subset=['source_id', 'camera_num']).drop_duplicates(['source_id', 'camera_num'])
    points = points.merge(raw2_small, on=['source_id', 'camera_num'], how='left')
else:
    points['expected_scatter'] = np.nan

points['finite_photometry'] = np.isfinite(points[['JD', 'mag', 'error']].to_numpy(dtype=float)).all(axis=1)
points['usable_point'] = (
    points['finite_photometry']
    & (pd.to_numeric(points['error'], errors='coerce') > 0)
    & (points['good_bad'].fillna(0).astype(int) == 1)
    & (points['saturated'].fillna(1).astype(int) == 0)
)

points['source_median_mag'] = points.groupby('source_id', observed=True)['mag'].transform('median')
points['source_band_median_mag'] = points.groupby(['source_id', 'band'], observed=True)['mag'].transform('median')
points['source_band_camera_median_mag'] = points.groupby(['source_id', 'band', 'camera_num'], observed=True)['mag'].transform('median')
points['diagnostic_baseline_mag'] = points['source_band_camera_median_mag'].fillna(points['source_band_median_mag']).fillna(points['source_median_mag'])
points['diagnostic_resid'] = points['mag'].astype(float) - points['diagnostic_baseline_mag'].astype(float)
points['abs_diagnostic_resid'] = points['diagnostic_resid'].abs()

camera_resid_sigma = points.groupby(['source_id', 'band', 'camera_num'], observed=True)['diagnostic_resid'].transform(robust_sigma)
source_band_resid_sigma = points.groupby(['source_id', 'band'], observed=True)['diagnostic_resid'].transform(robust_sigma)
source_resid_sigma = points.groupby('source_id', observed=True)['diagnostic_resid'].transform(robust_sigma)
resid_sigma = camera_resid_sigma.replace(0, np.nan).fillna(source_band_resid_sigma.replace(0, np.nan)).fillna(source_resid_sigma.replace(0, np.nan))
err_floor = pd.to_numeric(points['error'], errors='coerce').replace(0, np.nan)
points['diagnostic_resid_sigma'] = resid_sigma.fillna(err_floor).fillna(float(np.nanmedian(err_floor)))
points['norm_diagnostic_resid'] = points['diagnostic_resid'] / points['diagnostic_resid_sigma'].replace(0, np.nan)
points['tail_abs_norm_resid_gt4'] = points['norm_diagnostic_resid'].abs() > 4.0

candidate_projection.head()

## Field and Camera-Field Summary Tables

The rank score is a sorting aid assembled from transparent component columns. It is not a binary bad-field classifier.

In [ ]:
def group_summary(points_df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    agg = points_df.groupby(group_col, observed=True).agg(
        n_points=('mag', 'size'),
        n_sources=('source_id', 'nunique'),
        n_cameras=('camera_name', 'nunique'),
        n_camera_nums=('camera_num', 'nunique'),
        n_camera_fields=('camera_field', 'nunique'),
        jd_min=('JD', 'min'),
        jd_max=('JD', 'max'),
        finite_fraction=('finite_photometry', 'mean'),
        usable_fraction=('usable_point', 'mean'),
        saturated_fraction=('saturated', lambda s: float((pd.to_numeric(s, errors='coerce') == 1).mean()) if len(s) else np.nan),
        good_fraction=('good_bad', lambda s: float((pd.to_numeric(s, errors='coerce') == 1).mean()) if len(s) else np.nan),
        median_mag=('mag', 'median'),
        median_error=('error', 'median'),
        robust_scatter_mag=('mag', robust_sigma),
        robust_scatter_resid=('diagnostic_resid', robust_sigma),
        median_abs_resid=('abs_diagnostic_resid', 'median'),
        median_resid_offset=('diagnostic_resid', 'median'),
        p95_abs_norm_resid=('norm_diagnostic_resid', lambda s: float(np.nanpercentile(np.abs(pd.to_numeric(s, errors='coerce')), 95)) if len(s) else np.nan),
        max_abs_norm_resid=('norm_diagnostic_resid', lambda s: float(np.nanmax(np.abs(pd.to_numeric(s, errors='coerce')))) if len(s) else np.nan),
        tail_frac_gt4=('tail_abs_norm_resid_gt4', 'mean'),
        expected_scatter_median=('expected_scatter', 'median'),
    ).reset_index()
    agg['jd_span_days'] = agg['jd_max'] - agg['jd_min']
    agg['bad_quality_fraction'] = 1.0 - agg['usable_fraction']
    agg['scatter_expected_ratio'] = agg['robust_scatter_resid'] / agg['expected_scatter_median'].replace(0, np.nan)

    for value_col, count_name in [('camera_name', 'n_by_camera_name'), ('camera_field', 'n_by_camera_field'), ('band', 'n_by_band')]:
        if value_col == group_col:
            continue
        dom = top_value_fraction(points_df, group_col, value_col, count_name)
        agg = agg.merge(dom, on=group_col, how='left')
    return agg


field_summary = group_summary(points, 'field')
camera_field_summary = group_summary(points, 'camera_field')
if not camera_field_summary.empty:
    split_cf = camera_field_summary['camera_field'].astype(str).str.split('/', n=1, expand=True)
    camera_field_summary['camera_name'] = split_cf[0]
    camera_field_summary['field'] = split_cf[1] if split_cf.shape[1] > 1 else ''

source_field_summary = points.groupby(['source_id', 'field'], observed=True).agg(
    n_points=('mag', 'size'),
    n_field_cameras=('camera_name', 'nunique'),
    n_field_camera_fields=('camera_field', 'nunique'),
    jd_min=('JD', 'min'),
    jd_max=('JD', 'max'),
    usable_fraction=('usable_point', 'mean'),
    robust_scatter_resid=('diagnostic_resid', robust_sigma),
    median_abs_resid=('abs_diagnostic_resid', 'median'),
    median_resid_offset=('diagnostic_resid', 'median'),
    p95_abs_norm_resid=('norm_diagnostic_resid', lambda s: float(np.nanpercentile(np.abs(pd.to_numeric(s, errors='coerce')), 95)) if len(s) else np.nan),
    max_abs_norm_resid=('norm_diagnostic_resid', lambda s: float(np.nanmax(np.abs(pd.to_numeric(s, errors='coerce')))) if len(s) else np.nan),
    tail_frac_gt4=('tail_abs_norm_resid_gt4', 'mean'),
    expected_scatter_median=('expected_scatter', 'median'),
).reset_index()
source_field_summary['jd_span_days'] = source_field_summary['jd_max'] - source_field_summary['jd_min']
source_field_summary['scatter_expected_ratio'] = source_field_summary['robust_scatter_resid'] / source_field_summary['expected_scatter_median'].replace(0, np.nan)
source_field_summary = source_field_summary.merge(candidate_projection, on='source_id', how='left', suffixes=('', '_candidate'))

source_camera_field_summary = points.groupby(['source_id', 'camera_field'], observed=True).agg(
    n_points=('mag', 'size'),
    field=('field', lambda s: str(s.iloc[0]) if len(s) else ''),
    camera_name=('camera_name', lambda s: str(s.iloc[0]) if len(s) else ''),
    jd_min=('JD', 'min'),
    jd_max=('JD', 'max'),
    usable_fraction=('usable_point', 'mean'),
    robust_scatter_resid=('diagnostic_resid', robust_sigma),
    median_abs_resid=('abs_diagnostic_resid', 'median'),
    median_resid_offset=('diagnostic_resid', 'median'),
    p95_abs_norm_resid=('norm_diagnostic_resid', lambda s: float(np.nanpercentile(np.abs(pd.to_numeric(s, errors='coerce')), 95)) if len(s) else np.nan),
    max_abs_norm_resid=('norm_diagnostic_resid', lambda s: float(np.nanmax(np.abs(pd.to_numeric(s, errors='coerce')))) if len(s) else np.nan),
    tail_frac_gt4=('tail_abs_norm_resid_gt4', 'mean'),
    expected_scatter_median=('expected_scatter', 'median'),
).reset_index()
source_camera_field_summary['jd_span_days'] = source_camera_field_summary['jd_max'] - source_camera_field_summary['jd_min']
source_camera_field_summary['scatter_expected_ratio'] = source_camera_field_summary['robust_scatter_resid'] / source_camera_field_summary['expected_scatter_median'].replace(0, np.nan)
source_camera_field_summary = source_camera_field_summary.merge(candidate_projection, on='source_id', how='left', suffixes=('', '_candidate'))

display(field_summary.head())
display(camera_field_summary.head())
display(source_field_summary.head())

## Event-Window Field Dominance

This checks whether the strongest event window is dominated by one field/camera-field and whether other fields or camera-fields support the same event near the same time.

In [ ]:
def event_rows_from_candidates(candidate_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []
    for event_type in ['dip', 'jump']:
        t0_col = f'{event_type}_best_t0'
        width_col = f'{event_type}_best_width_param'
        sig_col = f'{event_type}_significant'
        run_col = f'{event_type}_run_count'
        if t0_col not in candidate_df.columns:
            continue
        keep_cols = ['source_id'] + [c for c in [t0_col, width_col, sig_col, run_col, f'{event_type}_bayes_factor', f'{event_type}_max_event_prob', f'{event_type}_max_run_cameras', f'{event_type}_best_morph'] if c in candidate_df.columns]
        subset = candidate_df[keep_cols].copy()
        subset = subset.rename(columns={
            t0_col: 'event_t0',
            width_col: 'event_width_param',
            sig_col: 'event_significant',
            run_col: 'event_run_count',
            f'{event_type}_bayes_factor': 'event_bayes_factor',
            f'{event_type}_max_event_prob': 'event_max_event_prob',
            f'{event_type}_max_run_cameras': 'event_max_run_cameras',
            f'{event_type}_best_morph': 'event_best_morph',
        })
        subset['event_type'] = event_type
        rows.append(subset)
    if not rows:
        return pd.DataFrame()
    events = pd.concat(rows, ignore_index=True)
    events['event_t0'] = pd.to_numeric(events['event_t0'], errors='coerce')
    events['event_width_param'] = pd.to_numeric(events.get('event_width_param', np.nan), errors='coerce')
    events['event_run_count'] = pd.to_numeric(events.get('event_run_count', 0), errors='coerce').fillna(0)
    events['event_significant'] = events.get('event_significant', events['event_run_count'] > 0)
    events['event_significant'] = pd.Series(events['event_significant']).fillna(False).astype(bool)
    return events[np.isfinite(events['event_t0']) & ((events['event_significant']) | (events['event_run_count'] > 0))].reset_index(drop=True)


def compute_event_field_dominance(points_df: pd.DataFrame, candidate_df: pd.DataFrame) -> pd.DataFrame:
    events = event_rows_from_candidates(candidate_df)
    if events.empty:
        return pd.DataFrame()
    source_scale = points_df.groupby('source_id', observed=True)['diagnostic_resid'].agg(robust_sigma).replace(0, np.nan).to_dict()
    grouped_points = {str(k): v.copy() for k, v in points_df.groupby('source_id', observed=True)}
    rows: list[dict] = []
    iterator = tqdm(events.itertuples(index=False), total=len(events), desc='Event dominance') if tqdm is not None else events.itertuples(index=False)
    for event in iterator:
        source_id = str(event.source_id)
        src = grouped_points.get(source_id)
        if src is None or src.empty:
            continue
        t0 = float(event.event_t0)
        width = float(event.event_width_param) if np.isfinite(event.event_width_param) and event.event_width_param > 0 else 2.0
        half_width = max(EVENT_WINDOW_SIGMA_MULT * width, 1.0)
        win = src[(src['JD'] >= t0 - half_width) & (src['JD'] <= t0 + half_width)].copy()
        if win.empty:
            continue
        event_type = str(event.event_type)
        direction = 1.0 if event_type == 'dip' else -1.0
        win['event_signal'] = direction * pd.to_numeric(win['diagnostic_resid'], errors='coerce')
        scale = source_scale.get(source_id, np.nan)
        signal_threshold = max(0.05, 2.0 * float(scale)) if np.isfinite(scale) else 0.05
        win['support_point'] = win['event_signal'] >= signal_threshold
        positive_signal = win['event_signal'].clip(lower=0).fillna(0)
        total_signal = float(positive_signal.sum())

        field_counts = win.groupby('field', observed=True).size().sort_values(ascending=False)
        camera_field_counts = win.groupby('camera_field', observed=True).size().sort_values(ascending=False)
        field_signal = positive_signal.groupby(win['field']).sum().sort_values(ascending=False)
        camera_field_signal = positive_signal.groupby(win['camera_field']).sum().sort_values(ascending=False)

        top_field = str(field_signal.index[0]) if total_signal > 0 and len(field_signal) else str(field_counts.index[0])
        top_camera_field = str(camera_field_signal.index[0]) if total_signal > 0 and len(camera_field_signal) else str(camera_field_counts.index[0])
        top_field_point_fraction = safe_fraction(field_counts.iloc[0], len(win)) if len(field_counts) else np.nan
        top_camera_field_point_fraction = safe_fraction(camera_field_counts.iloc[0], len(win)) if len(camera_field_counts) else np.nan
        top_field_signal_fraction = safe_fraction(field_signal.iloc[0], total_signal) if total_signal > 0 and len(field_signal) else np.nan
        top_camera_field_signal_fraction = safe_fraction(camera_field_signal.iloc[0], total_signal) if total_signal > 0 and len(camera_field_signal) else np.nan

        support_by_field = win.loc[win['support_point']].groupby('field', observed=True).size()
        support_by_camera_field = win.loc[win['support_point']].groupby('camera_field', observed=True).size()
        other_field_support = int(support_by_field.drop(labels=[top_field], errors='ignore').sum())
        other_camera_field_support = int(support_by_camera_field.drop(labels=[top_camera_field], errors='ignore').sum())

        dominated_by_field = bool(
            (np.nan_to_num(top_field_signal_fraction, nan=0.0) >= 0.80 or np.nan_to_num(top_field_point_fraction, nan=0.0) >= 0.80)
            and other_field_support == 0
            and win['field'].nunique() >= 2
        )
        dominated_by_camera_field = bool(
            (np.nan_to_num(top_camera_field_signal_fraction, nan=0.0) >= 0.80 or np.nan_to_num(top_camera_field_point_fraction, nan=0.0) >= 0.80)
            and other_camera_field_support == 0
            and win['camera_field'].nunique() >= 2
        )
        artifact_risk_score = float(
            max(np.nan_to_num(top_field_signal_fraction, nan=0.0), np.nan_to_num(top_camera_field_signal_fraction, nan=0.0))
            + 0.25 * dominated_by_field
            + 0.25 * dominated_by_camera_field
            - 0.10 * min(win['field'].nunique(), 5)
        )

        rows.append({
            'source_id': source_id,
            'event_type': event_type,
            'event_t0': t0,
            'event_half_width_days': half_width,
            'n_window_points': int(len(win)),
            'n_window_fields': int(win['field'].nunique()),
            'n_window_camera_fields': int(win['camera_field'].nunique()),
            'n_window_cameras': int(win['camera_name'].nunique()),
            'signal_threshold_mag': signal_threshold,
            'n_support_points': int(win['support_point'].sum()),
            'other_field_support_points': other_field_support,
            'other_camera_field_support_points': other_camera_field_support,
            'top_field': top_field,
            'top_camera_field': top_camera_field,
            'top_field_point_fraction': top_field_point_fraction,
            'top_camera_field_point_fraction': top_camera_field_point_fraction,
            'top_field_signal_fraction': top_field_signal_fraction,
            'top_camera_field_signal_fraction': top_camera_field_signal_fraction,
            'dominated_by_field_without_support': dominated_by_field,
            'dominated_by_camera_field_without_support': dominated_by_camera_field,
            'event_artifact_risk_score': artifact_risk_score,
            'event_bayes_factor': getattr(event, 'event_bayes_factor', np.nan),
            'event_max_event_prob': getattr(event, 'event_max_event_prob', np.nan),
            'event_max_run_cameras': getattr(event, 'event_max_run_cameras', np.nan),
            'event_best_morph': getattr(event, 'event_best_morph', ''),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.merge(candidate_projection, on='source_id', how='left', suffixes=('', '_candidate'))


event_field_dominance = compute_event_field_dominance(points, candidate_projection)
display(event_field_dominance.sort_values('event_artifact_risk_score', ascending=False).head(20) if not event_field_dominance.empty else event_field_dominance)

## Transparent Anomaly Components and Ranking Tables

Component columns are robust-z based and intentionally exposed. The `rank_score` is only a triage sorter.

In [ ]:
def add_review_association(summary: pd.DataFrame, group_col: str, source_group: pd.DataFrame) -> pd.DataFrame:
    if source_group.empty:
        return summary
    source_cols = [group_col, 'source_id'] + [c for c in ['bad_cameras_filtered', 'excluded_cameras', 'failed_any', 'dip_max_run_cameras', 'jump_max_run_cameras', 'interest_score', 'event_class', 'review_pass', 'status', 'microlens_match', 'vetting_likely_known'] if c in source_group.columns]
    assoc = source_group[source_cols].copy()
    assoc['has_bad_camera_filtered'] = assoc.get('bad_cameras_filtered', pd.Series('', index=assoc.index)).fillna('').astype(str).str.strip().ne('')
    assoc['has_excluded_cameras'] = assoc.get('excluded_cameras', pd.Series('', index=assoc.index)).fillna('').astype(str).str.strip().ne('')
    assoc['reviewed'] = assoc.get('interest_score', pd.Series(np.nan, index=assoc.index)).notna()
    group_assoc = assoc.groupby(group_col, observed=True).agg(
        candidate_sources=('source_id', 'nunique'),
        bad_camera_filtered_source_frac=('has_bad_camera_filtered', 'mean'),
        excluded_camera_source_frac=('has_excluded_cameras', 'mean'),
        reviewed_source_frac=('reviewed', 'mean'),
    ).reset_index()
    for col in ['dip_max_run_cameras', 'jump_max_run_cameras', 'interest_score']:
        if col in assoc.columns:
            extra = assoc.groupby(group_col, observed=True)[col].median().rename(f'{col}_median').reset_index()
            group_assoc = group_assoc.merge(extra, on=group_col, how='left')
    return summary.merge(group_assoc, on=group_col, how='left')


def add_anomaly_scores(summary: pd.DataFrame, group_col: str, event_col: str | None = None) -> pd.DataFrame:
    out = summary.copy()
    if event_col and not event_field_dominance.empty:
        event_agg = event_field_dominance.groupby(event_col).agg(
            n_events_as_top=('source_id', 'size'),
            n_field_dominated_events=('dominated_by_field_without_support', 'sum'),
            n_camera_field_dominated_events=('dominated_by_camera_field_without_support', 'sum'),
            max_event_artifact_risk_score=('event_artifact_risk_score', 'max'),
            median_event_artifact_risk_score=('event_artifact_risk_score', 'median'),
        ).reset_index().rename(columns={event_col: group_col})
        out = out.merge(event_agg, on=group_col, how='left')
    for col in ['n_events_as_top', 'n_field_dominated_events', 'n_camera_field_dominated_events', 'max_event_artifact_risk_score', 'median_event_artifact_risk_score']:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0.0)

    out['coverage_low_source_z'] = positive_robust_z(-np.log1p(pd.to_numeric(out['n_sources'], errors='coerce')))
    out['coverage_low_point_z'] = positive_robust_z(-np.log1p(pd.to_numeric(out['n_points'], errors='coerce')))
    out['coverage_narrow_span_z'] = positive_robust_z(-np.log1p(pd.to_numeric(out['jd_span_days'], errors='coerce').clip(lower=0)))
    out['coverage_one_camera_z'] = positive_robust_z(out.get('camera_name_dominance_fraction', pd.Series(0, index=out.index)).fillna(0))
    out['coverage_one_band_z'] = positive_robust_z(out.get('band_dominance_fraction', pd.Series(0, index=out.index)).fillna(0))
    out['coverage_bad_quality_z'] = positive_robust_z(out['bad_quality_fraction'].fillna(0))
    out['coverage_component'] = out[[
        'coverage_low_source_z', 'coverage_low_point_z', 'coverage_narrow_span_z',
        'coverage_one_camera_z', 'coverage_one_band_z', 'coverage_bad_quality_z'
    ]].max(axis=1)

    out['phot_scatter_z'] = positive_robust_z(out['robust_scatter_resid'])
    out['phot_error_z'] = positive_robust_z(out['median_error'])
    out['phot_offset_abs_z'] = positive_robust_z(out['median_resid_offset'].abs())
    out['phot_tail_frac_z'] = positive_robust_z(out['tail_frac_gt4'].fillna(0))
    out['phot_max_abs_norm_z'] = positive_robust_z(out['max_abs_norm_resid'])
    out['phot_expected_ratio_z'] = positive_robust_z(out['scatter_expected_ratio'].replace([np.inf, -np.inf], np.nan))
    out['photometric_component'] = out[[
        'phot_scatter_z', 'phot_error_z', 'phot_offset_abs_z',
        'phot_tail_frac_z', 'phot_max_abs_norm_z', 'phot_expected_ratio_z'
    ]].max(axis=1)

    out['event_component'] = positive_robust_z(out['max_event_artifact_risk_score'].fillna(0))
    out['low_n_caveat'] = pd.to_numeric(out['n_points'], errors='coerce') < MIN_GROUP_POINTS
    out['rank_score'] = (
        out['coverage_component'].clip(upper=8)
        + out['photometric_component'].clip(upper=8)
        + out['event_component'].clip(upper=8)
        + 0.5 * out['low_n_caveat'].astype(float)
    )
    return out.sort_values(['rank_score', 'photometric_component', 'event_component'], ascending=False).reset_index(drop=True)


field_summary = add_review_association(field_summary, 'field', source_field_summary)
camera_field_summary = add_review_association(camera_field_summary, 'camera_field', source_camera_field_summary)
field_summary = add_anomaly_scores(field_summary, 'field', event_col='top_field')
camera_field_summary = add_anomaly_scores(camera_field_summary, 'camera_field', event_col='top_camera_field')

source_field_for_rank = source_field_summary.copy()
source_field_for_rank['n_sources'] = 1
source_field_for_rank['n_camera_nums'] = source_field_for_rank['n_field_cameras']
source_field_for_rank['bad_quality_fraction'] = 1.0 - source_field_for_rank['usable_fraction']
source_field_for_rank['finite_fraction'] = np.nan
source_field_for_rank['median_error'] = np.nan
source_field_for_rank['robust_scatter_mag'] = np.nan
source_field_for_rank['field'] = source_field_for_rank['field'].astype(str)
source_field_for_rank = add_anomaly_scores(source_field_for_rank, 'field', event_col=None)

event_risk_by_source = event_field_dominance.groupby('source_id')['event_artifact_risk_score'].max().rename('max_event_artifact_risk_score_by_source') if not event_field_dominance.empty else pd.Series(dtype=float)
top_suspect_candidates = (
    source_field_for_rank.groupby('source_id', observed=True)
    .agg(
        max_source_field_rank_score=('rank_score', 'max'),
        max_source_field_photometric_component=('photometric_component', 'max'),
        max_source_field_coverage_component=('coverage_component', 'max'),
        worst_field=('field', lambda s: s.iloc[0] if len(s) else ''),
    )
    .reset_index()
    .merge(candidate_projection, on='source_id', how='left')
)
if not event_risk_by_source.empty:
    top_suspect_candidates = top_suspect_candidates.merge(event_risk_by_source.reset_index(), on='source_id', how='left')
else:
    top_suspect_candidates['max_event_artifact_risk_score_by_source'] = np.nan
top_suspect_candidates['candidate_rank_score'] = (
    top_suspect_candidates['max_source_field_rank_score'].fillna(0)
    + positive_robust_z(top_suspect_candidates['max_event_artifact_risk_score_by_source'].fillna(0)).clip(upper=8)
)
top_suspect_candidates = top_suspect_candidates.sort_values('candidate_rank_score', ascending=False).reset_index(drop=True)

top_suspect_fields = field_summary.head(50)
top_suspect_camera_fields = camera_field_summary.head(50)

display_section('Top Suspect Fields', 'Ranking aid only; inspect components and caveats before interpreting.')
display(top_suspect_fields[['field', 'rank_score', 'coverage_component', 'photometric_component', 'event_component', 'n_points', 'n_sources', 'n_cameras', 'camera_name_dominance_fraction', 'band_dominance_fraction', 'robust_scatter_resid', 'median_error', 'scatter_expected_ratio', 'tail_frac_gt4', 'max_event_artifact_risk_score', 'low_n_caveat']].head(25))

display_section('Top Suspect Camera-Fields')
display(top_suspect_camera_fields[['camera_field', 'field', 'camera_name', 'rank_score', 'coverage_component', 'photometric_component', 'event_component', 'n_points', 'n_sources', 'robust_scatter_resid', 'median_error', 'scatter_expected_ratio', 'tail_frac_gt4', 'max_event_artifact_risk_score', 'low_n_caveat']].head(25))

display_section('Top Suspect Candidates')
candidate_display_cols = [c for c in ['source_id', 'candidate_rank_score', 'max_source_field_rank_score', 'max_event_artifact_risk_score_by_source', 'worst_field', 'dip_max_run_cameras', 'jump_max_run_cameras', 'bad_cameras_filtered', 'excluded_cameras', 'event_class', 'interest_score', 'review_pass'] if c in top_suspect_candidates.columns]
display(top_suspect_candidates[candidate_display_cols].head(25))

## Visual Diagnostics

These plots focus on distributions and concentrations, not final labels.

In [ ]:
def mark_robust_threshold(ax, values: pd.Series, label: str, threshold_z: float = ROBUST_Z_THRESHOLD) -> None:
    vals = pd.to_numeric(values, errors='coerce')
    finite = vals[np.isfinite(vals)]
    if finite.empty:
        return
    med = float(np.nanmedian(finite))
    scale = robust_sigma(finite)
    if np.isfinite(scale) and scale > 0:
        x = med + threshold_z * scale
        ax.axvline(x, color='crimson', linestyle='--', linewidth=1.5, label=label)
        ax.legend()


fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()
axes[0].hist(np.log10(field_summary['n_points'].clip(lower=1)), bins=40, color='#2b6f9e', alpha=0.8)
axes[0].set_title('Field point counts')
axes[0].set_xlabel('log10(n_points)')
axes[1].hist(np.log10(camera_field_summary['n_points'].clip(lower=1)), bins=40, color='#b46b35', alpha=0.8)
axes[1].set_title('Camera-field point counts')
axes[1].set_xlabel('log10(n_points)')
axes[2].hist(field_summary['n_sources'].clip(lower=1), bins=40, color='#4c8c44', alpha=0.8)
axes[2].set_title('Sources per field')
axes[2].set_xlabel('n_sources within candidate bundle')
axes[3].hist(camera_field_summary['n_sources'].clip(lower=1), bins=40, color='#7b5aa6', alpha=0.8)
axes[3].set_title('Sources per camera-field')
axes[3].set_xlabel('n_sources within candidate bundle')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
metric_specs = [
    ('robust_scatter_resid', 'Robust residual scatter'),
    ('median_error', 'Median reported error'),
    ('median_resid_offset', 'Median residual offset'),
    ('tail_frac_gt4', '|norm residual| > 4 fraction'),
    ('scatter_expected_ratio', 'Scatter / raw2 expected scatter'),
    ('rank_score', 'Triage rank score'),
]
for ax, (metric, title) in zip(axes.ravel(), metric_specs):
    vals = pd.to_numeric(camera_field_summary[metric], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(vals, bins=50, color='#425466', alpha=0.85)
    mark_robust_threshold(ax, vals, f'+{ROBUST_Z_THRESHOLD:g} robust sigma')
    ax.set_title(title)
    ax.set_xlabel(metric)
plt.tight_layout()
plt.show()

heat = camera_field_summary.copy()
heat = heat[heat['n_points'] >= MIN_GROUP_POINTS].copy()
top_fields_for_heat = heat.sort_values('rank_score', ascending=False)['field'].drop_duplicates().head(25)
top_cameras_for_heat = heat.sort_values('rank_score', ascending=False)['camera_name'].drop_duplicates().head(20)
heat_small = heat[heat['field'].isin(top_fields_for_heat) & heat['camera_name'].isin(top_cameras_for_heat)]
if not heat_small.empty:
    pivot = heat_small.pivot_table(index='camera_name', columns='field', values='rank_score', aggfunc='max')
    fig, ax = plt.subplots(figsize=(max(12, 0.45 * pivot.shape[1]), max(6, 0.35 * pivot.shape[0])))
    im = ax.imshow(pivot.fillna(0).to_numpy(), aspect='auto', cmap='magma')
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_title('Camera vs field anomaly rank score (top combinations only)')
    fig.colorbar(im, ax=ax, label='rank_score')
    plt.tight_layout()
    plt.show()
else:
    display(Markdown('No heatmap: no camera-field rows after filtering.'))

if not event_field_dominance.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    event_field_dominance['top_field_point_fraction'].hist(ax=axes[0], bins=30, color='#476c9b')
    axes[0].set_title('Event-window top field point fraction')
    event_field_dominance['top_field_signal_fraction'].dropna().hist(ax=axes[1], bins=30, color='#a45d5d')
    axes[1].set_title('Event-window top field signal fraction')
    event_field_dominance['event_artifact_risk_score'].hist(ax=axes[2], bins=30, color='#6b7f3a')
    axes[2].set_title('Event artifact risk score')
    for ax in axes:
        ax.set_ylabel('events')
    plt.tight_layout()
    plt.show()

    display(event_field_dominance.sort_values('event_artifact_risk_score', ascending=False).head(25))
else:
    display(Markdown('No event dominance rows were produced.'))

## Example Source Panels

These plots color each source by `camera_field` and mark the best dip/jump times when available. They are designed for visual triage of the highest-ranked candidate-level cases.

In [ ]:
def plot_source_lightcurve(source_id: str, points_df: pd.DataFrame, candidate_df: pd.DataFrame, ax=None) -> None:
    src = points_df[points_df['source_id'].astype(str) == str(source_id)].copy()
    if src.empty:
        return
    cand = candidate_df[candidate_df['source_id'].astype(str) == str(source_id)].head(1)
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 4))
    labels = src['camera_field'].astype(str)
    unique_labels = labels.drop_duplicates().tolist()
    cmap = plt.get_cmap('tab20')
    color_map = {label: cmap(i % 20) for i, label in enumerate(unique_labels)}
    for label, sub in src.groupby(labels, observed=False):
        ax.scatter(sub['JD'], sub['mag'], s=9, alpha=0.72, color=color_map[str(label)], label=str(label))
    if not cand.empty:
        row = cand.iloc[0]
        for event_type, color in [('dip', 'crimson'), ('jump', 'seagreen')]:
            t0 = pd.to_numeric(pd.Series([row.get(f'{event_type}_best_t0')]), errors='coerce').iloc[0]
            width = pd.to_numeric(pd.Series([row.get(f'{event_type}_best_width_param')]), errors='coerce').iloc[0]
            if np.isfinite(t0):
                width = width if np.isfinite(width) and width > 0 else 2.0
                half_width = max(EVENT_WINDOW_SIGMA_MULT * float(width), 1.0)
                ax.axvline(float(t0), color=color, linewidth=1.6, alpha=0.85)
                ax.axvspan(float(t0) - half_width, float(t0) + half_width, color=color, alpha=0.08)
    ax.invert_yaxis()
    ax.set_title(f'{source_id} colored by camera_field')
    ax.set_xlabel('JD')
    ax.set_ylabel('mag')
    if len(unique_labels) <= 12:
        ax.legend(loc='best', fontsize=7, frameon=False)


example_sources = top_suspect_candidates['source_id'].dropna().astype(str).drop_duplicates().head(MAX_EXAMPLE_PLOTS).tolist()
if example_sources:
    for source_id in example_sources:
        fig, ax = plt.subplots(figsize=(13, 4))
        plot_source_lightcurve(source_id, points, candidate_projection, ax=ax)
        plt.tight_layout()
        plt.show()
else:
    display(Markdown('No example sources available.'))

## Optional Cache of Derived Tables

This is disabled by default. Enable `CACHE_DERIVED_TABLES` in the parameter cell to write reusable parquet outputs.

In [ ]:
derived_tables = {
    'field_summary': field_summary,
    'camera_field_summary': camera_field_summary,
    'source_field_summary': source_field_summary,
    'source_camera_field_summary': source_camera_field_summary,
    'event_field_dominance': event_field_dominance,
    'top_suspect_fields': top_suspect_fields,
    'top_suspect_camera_fields': top_suspect_camera_fields,
    'top_suspect_candidates': top_suspect_candidates,
    'integrity_summary': integrity_summary,
    'parse_issues': parse_issues,
    'raw2_issues': raw2_issues,
}

if CACHE_DERIVED_TABLES:
    analysis_dir = BUNDLE_ROOT / 'analysis/field_anomalies'
    analysis_dir.mkdir(parents=True, exist_ok=True)
    for name, table in derived_tables.items():
        if isinstance(table, pd.DataFrame):
            table.to_parquet(analysis_dir / f'{name}.parquet', index=False)
    with (analysis_dir / 'analysis_params.json').open('w') as f:
        json.dump({
            'FULL_SCAN': FULL_SCAN,
            'MAX_EXAMPLE_PLOTS': MAX_EXAMPLE_PLOTS,
            'EVENT_WINDOW_SIGMA_MULT': EVENT_WINDOW_SIGMA_MULT,
            'MIN_GROUP_POINTS': MIN_GROUP_POINTS,
            'ROBUST_Z_THRESHOLD': ROBUST_Z_THRESHOLD,
            'bundle_root': str(BUNDLE_ROOT),
        }, f, indent=2)
    display(Markdown(f'Cached derived tables to `{analysis_dir}`.'))
else:
    display(Markdown('`CACHE_DERIVED_TABLES` is False; no derived tables were written.'))

## Findings Template

Fill this section after reviewing the ranked tables and plots.

### Top recurrent anomalous fields
- Field:
- Evidence:
- Caveat:
- Follow-up:

### Top anomalous camera-fields
- Camera-field:
- Evidence:
- Caveat:
- Follow-up:

### Candidates whose event evidence is field-dominated
- Source ID:
- Event type:
- Dominating field/camera-field:
- Other-field support:
- Interpretation:

### Candidates likely safer from field artifacts
- Source ID:
- Event type:
- Supporting fields/camera-fields:
- Interpretation:

### Explicit caveats
- This notebook analyzes only the March 18 vetted/passing candidate bundle, not the full ASAS-SN source population.
- Field-level rates here are within-bundle concentration diagnostics, not survey-wide false-positive rates.
- Diagnostic residuals are median-based and do not exactly reproduce MALCA GP baseline residuals.
- `.raw2` scatter expectations are camera-level, not field-level, so `scatter_expected_ratio` is an approximate diagnostic.

### Recommended next actions
- Manually inspect top field-dominated events in the review UI or source panels.
- Compare top suspect fields against a full prefilter or all-light-curve denominator before adopting any field rejection rule.
- If recurrent artifacts are confirmed, propagate `field`/`camera_field` into production outputs and add diagnostics before filtering.